# Milestone 3: Classical Baselines vs RL on the CVD Digital Twin

Goal. Benchmark a reinforcement learning (RL) agent (DQN) against simple, interpretable classical/manual strategies on a CVD digital twin and report Milestone-3 metrics: final error, control effort, stability (reversals + overshoot), and steps.

## Experimental setup

Env. CVDReactorEnv (Arrhenius growth). Output rate is scaled to nm/s; thickness is nm.
Target thickness: 100 nm. Episode horizon: 50 steps.

Actions. Discrete changes in ΔT and ΔF.

Determinism. No process noise in M3. Fixed seed for env, action space, NumPy, Python random, and SB3.

Policies.

Always-Max

Proportional

Random

RL (DQN) — 5k training steps

Step-Ramp: ramp T to ~80% of target-thickness progress, then hold T while nudging F, then hold.

## Metrics (per episode)

error = |h_final − h_target| (nm)

effort = Σ(|ΔT| + |ΔF|)

overshoot = max(0, max(h) − h_target) (nm)

reversals = sign-changes in ΔT and ΔF after reaching 0.5·h_target

steps = episode length


In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd

from cvd_env.reactor_env import CVDReactorEnv
from agents.manual_policies import (
    always_max_policy,
    proportional_policy,
    random_policy,
    step_ramp_policy,
)
from agents.dqn_trainer import train_dqn
from agents.compare_policies import evaluate_policy, summarize_metrics
from utils.logger import setup_logging, get_logger
from stable_baselines3.common.utils import set_random_seed

setup_logging()
logger = get_logger(__name__)


SEED = 42
np.random.seed(SEED)
random.seed(SEED)
set_random_seed(SEED)

In [ ]:
def make_env():
    e = CVDReactorEnv(
        target_thickness=100.0,
        max_steps=50,
        k0=3.16e15,
        Ea=237794,
        alpha=1.0,
        dt=0.01,
        mode="real",
        P=1e5,
        V=1e-3,
    )
    e.action_space.seed(SEED)
    e.reset(seed=SEED)
    return e


env = make_env()

In [ ]:
model = train_dqn(env, total_timesteps=5000)


def rl_policy(env, obs):
    action, _ = model.predict(obs, deterministic=True)
    return int(action)

In [ ]:
n_episodes = 10

metrics_max = evaluate_policy(always_max_policy, make_env(), n_episodes)
metrics_prop = evaluate_policy(proportional_policy, make_env(), n_episodes)
metrics_rand = evaluate_policy(
    lambda env, s: random_policy(env), make_env(), n_episodes
)
metrics_rl = evaluate_policy(rl_policy, make_env(), n_episodes)
metrics_step = evaluate_policy(step_ramp_policy, make_env(), n_episodes)

In [ ]:
from IPython.display import display


def to_series(name, metrics):
    s = summarize_metrics(metrics)
    flat = {f"{k} (mean)": v["mean"] for k, v in s.items()} | {
        f"{k} (std)": v["std"] for k, v in s.items()
    }
    return pd.Series(flat, name=name)


rows = [
    to_series("Always Max", metrics_max),
    to_series("Proportional", metrics_prop),
    to_series("Random", metrics_rand),
    to_series("RL (DQN)", metrics_rl),
    to_series("Step Ramp", metrics_step),
]

summary_df = pd.concat(rows, axis=1).T
summary_df = summary_df[
    [
        "error (mean)",
        "error (std)",
        "overshoot (mean)",
        "overshoot (std)",
        "reversals (mean)",
        "reversals (std)",
        "effort (mean)",
        "effort (std)",
        "steps (mean)",
        "steps (std)",
    ]
]
display(summary_df.style.format("{:.2f}"))

In [ ]:
def stack_metrics(name, metrics):
    df = pd.DataFrame(metrics)
    df["Policy"] = name
    return df


df_all = pd.concat(
    [
        stack_metrics("Always Max", metrics_max),
        stack_metrics("Proportional", metrics_prop),
        stack_metrics("Random", metrics_rand),
        stack_metrics("RL (DQN)", metrics_rl),
        stack_metrics("Step Ramp", metrics_step),
    ],
    ignore_index=True,
)

for metric in ["error", "overshoot", "reversals", "effort", "steps"]:
    plt.figure(figsize=(6, 4))
    df_all.boxplot(column=metric, by="Policy")
    plt.title(f"{metric.capitalize()} by Policy")
    plt.suptitle("")
    plt.xlabel("Policy")
    plt.ylabel(metric.capitalize())
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
from utils.plots import plot_trajectories


def run_and_plot_trajectory(policy_fn, env, policy_name):
    obs, _ = env.reset(seed=SEED)
    done = False
    thickness = [obs[0]]
    temperature = [obs[1]]
    flow_rate = [obs[2]]

    while not done:
        action = policy_fn(env, obs)
        obs, _, terminated, truncated, info = env.step(action)
        thickness.append(obs[0])
        temperature.append(obs[1])
        flow_rate.append(obs[2])
        done = terminated or truncated

    plot_trajectories(
        thickness,
        temperature,
        flow_rate,
        env.h_target,
        title=f"{policy_name} Policy Trajectory",
    )


run_and_plot_trajectory(
    always_max_policy,
    CVDReactorEnv(
        target_thickness=100.0,
        max_steps=50,
        k0=3.16e15,
        Ea=237794,
        alpha=1.0,
        dt=0.01,
        mode="real",
        P=1e5,
        V=1e-3,
    ),
    "Always Max",
)
run_and_plot_trajectory(
    proportional_policy,
    CVDReactorEnv(
        target_thickness=100.0,
        max_steps=50,
        k0=3.16e15,
        Ea=237794,
        alpha=1.0,
        dt=0.01,
        mode="real",
        P=1e5,
        V=1e-3,
    ),
    "Proportional",
)
run_and_plot_trajectory(
    lambda env, obs: random_policy(env),
    CVDReactorEnv(
        target_thickness=100.0,
        max_steps=50,
        k0=3.16e15,
        Ea=237794,
        alpha=1.0,
        dt=0.01,
        mode="real",
        P=1e5,
        V=1e-3,
    ),
    "Random",
)
run_and_plot_trajectory(
    rl_policy,
    CVDReactorEnv(
        target_thickness=100.0,
        max_steps=50,
        k0=3.16e15,
        Ea=237794,
        alpha=1.0,
        dt=0.01,
        mode="real",
        P=1e5,
        V=1e-3,
    ),
    "RL (DQN)",
)
run_and_plot_trajectory(
    step_ramp_policy,
    CVDReactorEnv(
        target_thickness=100.0,
        max_steps=50,
        k0=3.16e15,
        Ea=237794,
        alpha=1.0,
        dt=0.01,
        mode="real",
        P=1e5,
        V=1e-3,
    ),
    "Step Ramp",
)

## Results

### Full metrics (10 episodes per policy; mean ± std)

| Policy       |        Error (nm) |   Overshoot (nm) |       Reversals |             Effort |            Steps |
| ------------ | ----------------: | ---------------: | --------------: | -----------------: | ---------------: |
| Always Max   |  **39.64 ± 0.00** | **39.64 ± 0.00** | **0.00 ± 0.00** |  **105.00 ± 0.00** |  **7.00 ± 0.00** |
| Proportional |  **36.33 ± 0.00** | **36.33 ± 0.00** | **0.00 ± 0.00** |  **100.00 ± 0.00** |  **7.00 ± 0.00** |
| Random       | **29.36 ± 26.29** |  **1.14 ± 1.74** | **9.40 ± 8.48** | **458.00 ± 91.33** | **45.80 ± 8.08** |
| RL (DQN)     |  **30.84 ± 0.00** | **30.84 ± 0.00** | **0.00 ± 0.00** |   **95.00 ± 0.00** |  **7.00 ± 0.00** |
| Step Ramp    |  **26.36 ± 0.00** | **26.36 ± 0.00** | **0.00 ± 0.00** |   **75.00 ± 0.00** |  **8.00 ± 0.00** |

_Zero standard deviation indicates deterministic behavior under fixed seeds and no process noise.
In these seeded runs, final error equals overshoot for deterministic policies because episodes terminate as soon as thickness exceeds the target._

(See boxplots for Error, Overshoot, Reversals, Effort, and Steps; plus per-policy trajectory plots.)

---

## Interpretation & Discussion

- Step-Ramp is best on precision and cost. It achieves the lowest error and lowest control effort among the non-random policies, taking ~8 steps and landing close to target without oscillation.
- RL (DQN @ 5k steps) is stable and efficient (effort 95, fewer moves than Always-Max/Proportional) but not yet as precise as Step-Ramp (error 30.84 vs 26.36).
- Always-Max/Proportional are fast but sloppy: they hit the target quickly (7 steps) but consistently overshoot, so their final error is high.
- Random sometimes finishes near target (low average overshoot) but is unreliable (many reversals), expensive (very high effort), and slow (often runs to the horizon).

**Why RL isn’t winning yet**

With only 5k training steps and a coarse discrete action space, the agent hasn’t fully learned the “ramp-then-feather” behavior. Reward only penalizes final error, so it doesn’t explicitly prefer soft landings or low effort.

**Trajectory insights**

Deterministic policies climb thickness monotonically; Step-Ramp feathers actions near the target; RL shows a similar shape but still lands slightly high; Random meanders with frequent direction changes.

---

## Next Steps

- Train longer (≥ 50k steps) and/or switch to continuous control (PPO/SAC) for finer actuation.
- Reward shaping: penalize overshoot and large/late moves to encourage soft landings and lower effort.
- Try finer action discretization or continuous control algorithms (e.g., PPO, SAC).
- Add a time-to-target (steps-to-spec) metric using a tolerance band (e.g., ±1 nm) to capture throughput.
- In M4, introduce process/sensor noise and gas-phase loss; re-benchmark—RL’s adaptability should widen the gap vs heuristics.

---

## Summary

- Milestone-3 requirements are met with the upgraded evaluator (error, effort, overshoot, reversals, steps), seeded runs, and the new Step-Ramp baseline.
- Current leaderboard: Step-Ramp (best error/effort) > RL (stable/efficient, needs more training) > Proportional/Always-Max (fast but overshoot) > Random (unreliable).
- This sets the stage for M4 physics and longer RL training to surpass the heuristic ramp.
